# Executive Sentiment — Exploratory Data Analysis and Cleaning
**WUSS Academic Journal Committee · Spring 2026**

This notebook explores the output of the Track A scoring pipeline. The goal is to understand the dataset's coverage, score distributions, and quality before handing off to Track B for volatility computation and regression analysis.

**Input files (loaded directly from GitHub):**
- `strux_scores.csv` — sentiment and uncertainty scores for all transcripts
- `strux_manifest.csv` — ticker and date pairs

**Key questions:**
- How many transcripts and tickers do we have?
- Are the score distributions sensible?
- Which tickers are well-represented enough for panel regression?
- Is there meaningful sector coverage?

## 1. Setup and Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm import tqdm

## 2. Load Data

We load directly from the GitHub repository so this notebook runs without any local file dependencies.

In [ ]:
scores_url = "https://raw.githubusercontent.com/darrentweng/wharton-uss-academic-journal-executive-sentiment/refs/heads/main/data/strux_scores.csv"
manifest_url = "https://raw.githubusercontent.com/darrentweng/wharton-uss-academic-journal-executive-sentiment/refs/heads/main/data/strux_manifest.csv"

df = pd.read_csv(scores_url)
df['date'] = pd.to_datetime(df['date'])

manifest = pd.read_csv(manifest_url)
manifest['date'] = pd.to_datetime(manifest['date'])

print(f"Scores loaded: {len(df)} rows, {df['ticker'].nunique()} unique tickers")
print(f"Manifest loaded: {len(manifest)} rows")
print(f"\nDate range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nColumns: {list(df.columns)}")

## 3. Score Distributions

We expect:
- `sentiment_score` to be moderately positive (earnings calls skew optimistic), roughly centered around 0.3 to 0.5
- `uncertainty_score` to be a small positive fraction, typically 0.005 to 0.015
- Q&A scores to differ from prepared remarks — executives are less scripted when responding to analysts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Score Distributions — All Transcripts", fontsize=14)

df['sentiment_score'].hist(bins=50, ax=axes[0,0], color='steelblue', edgecolor='white')
axes[0,0].set_title("Sentiment Score (Prepared Remarks)")
axes[0,0].set_xlabel("Score")

df['uncertainty_score'].hist(bins=50, ax=axes[0,1], color='darkorange', edgecolor='white')
axes[0,1].set_title("Uncertainty Score (Prepared Remarks)")
axes[0,1].set_xlabel("Score")

df['sentiment_score_qa'].dropna().hist(bins=50, ax=axes[1,0], color='steelblue', edgecolor='white', alpha=0.7)
axes[1,0].set_title("Sentiment Score (Q&A)")
axes[1,0].set_xlabel("Score")

df['uncertainty_score_qa'].dropna().hist(bins=50, ax=axes[1,1], color='darkorange', edgecolor='white', alpha=0.7)
axes[1,1].set_title("Uncertainty Score (Q&A)")
axes[1,1].set_xlabel("Score")

plt.tight_layout()
plt.savefig("score_distributions.png", dpi=150)
plt.show()

print(df[['sentiment_score', 'uncertainty_score',
          'sentiment_score_qa', 'uncertainty_score_qa']].describe().round(6))

## 4. Score Observations

The distributions confirm expected behavior:

- Prepared remarks sentiment is right-skewed and moderately positive (mean ~0.40), consistent with the expectation that executives communicate optimistically in scripted speech.
- Q&A sentiment is noticeably lower (mean ~0.20), suggesting executives are more guarded when responding to analyst questions under pressure. This gap is substantively interesting and will be tested as a supplementary regression.
- Uncertainty scores are small positive fractions (mean ~0.008 for prepared remarks, ~0.008 for Q&A), which is expected given that hedging words are a small fraction of total transcript vocabulary.
- No extreme sentiment scores (>0.9 or <-0.5) were found, indicating no obvious scoring failures.

## 5. Ticker Coverage

We need to understand how many transcripts each ticker has. For a fixed-effects panel regression, we need enough within-firm observations to estimate meaningful time-series variation. Tickers with very few transcripts contribute little to the regression and may introduce noise.

In [ ]:
counts = df['ticker'].value_counts()

print("Transcripts per ticker:")
print(f"  Mean:   {counts.mean():.1f}")
print(f"  Median: {counts.median():.1f}")
print(f"  Min:    {counts.min()}")
print(f"  Max:    {counts.max()}")
print(f"\nTotal tickers: {len(counts)}")
print(f"Tickers with fewer than 9 transcripts: {(counts < 9).sum()} ({(counts < 9).mean()*100:.1f}%)")

## 6. Coverage Distributions

The left chart shows how many tickers fall into each transcript count bucket. The right chart shows how many transcripts were published per quarter across the full dataset.

The transcript-per-quarter chart reveals two important features:
- Coverage is sparse before 2018, reflecting limited transcript availability in the Strux dataset for early years.
- There is a large spike in early 2024, which appears to be a structural artifact of how the Strux dataset was assembled rather than a genuine increase in earnings call frequency. This motivates filtering the regression sample to 2018 through 2023.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts.hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title("Transcripts per Ticker")
axes[0].set_xlabel("Number of transcripts")
axes[0].set_ylabel("Number of tickers")

df.groupby(df['date'].dt.to_period('Q')).size().plot(ax=axes[1], color='steelblue')
axes[1].set_title("Transcripts per Quarter")
axes[1].set_xlabel("Quarter")
axes[1].set_ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Sector Labels

We fetch GICS sector labels from yfinance for each ticker. This takes a few minutes to run. Sector labels are used to understand the composition of the dataset and to assess whether sector-level subgroup analysis is viable.

In [ ]:
sector_map = {}
for ticker in tqdm(df['ticker'].unique()):
    try:
        info = yf.Ticker(ticker).info
        sector_map[ticker] = info.get('sector', 'Unknown')
    except:
        sector_map[ticker] = 'Unknown'

df['sector'] = df['ticker'].map(sector_map)
print(df['sector'].value_counts())

## 8. Sector Coverage — Full Dataset

The left chart shows unique tickers per sector. The right chart shows the distribution of sentiment scores by sector.

Sentiment medians are remarkably consistent across sectors (all around 0.40), which is consistent with the general finding in the textual analysis literature that earnings calls skew positive regardless of industry. The consistency across sectors also supports the use of fixed effects — between-sector variation in sentiment is low, so the regression signal is coming from within-firm variation over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('sector')['ticker'].nunique().sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title("Unique Tickers per Sector (Full Dataset)")

df.boxplot(column='sentiment_score', by='sector', ax=axes[1], rot=45)
axes[1].set_title("Sentiment Score by Sector (Full Dataset)")
axes[1].set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 9. Data Quality Checks

We check for nulls, extreme sentiment scores (potential scoring failures), and zero uncertainty scores (potential empty transcripts). Any problematic rows will be excluded from the clean dataset.

In [ ]:
print("Null counts:")
print(df[['sentiment_score', 'uncertainty_score',
          'sentiment_score_qa', 'uncertainty_score_qa']].isnull().sum())

extreme = df[(df['sentiment_score'] > 0.9) | (df['sentiment_score'] < -0.5)]
print(f"\nExtreme sentiment scores (>0.9 or <-0.5): {len(extreme)}")
if len(extreme) > 0:
    print(extreme[['ticker', 'date', 'sentiment_score']].head(10))

zero_uncertainty = df[df['uncertainty_score'] == 0]
print(f"\nZero uncertainty score: {len(zero_uncertainty)}")
if len(zero_uncertainty) > 0:
    print(zero_uncertainty[['ticker', 'date']].head(10))

## 10. Constructing the Clean Dataset

We apply two filters in sequence to construct the regression-ready dataset:

1. **Date filter:** Restrict to 2018 through 2023. Pre-2018 coverage is sparse and the 2024 spike is a dataset artifact. This filter is applied first.
2. **Coverage filter:** Keep only tickers with 9 or more transcripts within the filtered date range. This threshold is a judgment call — it ensures enough within-firm observations for fixed-effects estimation while retaining as many tickers as possible. The sensitivity of results to alternative thresholds (6, 12) will be reported as a robustness check.

The order matters: filtering by date first ensures the coverage threshold is applied to the actual regression window, not the full dataset.

In [ ]:
# Step 1: date filter
df_dated = df[(df['date'] >= '2018-01-01') & (df['date'] <= '2023-12-31')].copy()
print(f"After date filter: {len(df_dated)} transcripts, {df_dated['ticker'].nunique()} tickers")

# Step 2: recount within date range
counts_dated = df_dated['ticker'].value_counts()

# Step 3: apply coverage threshold
well_represented = counts_dated[counts_dated >= 9].index
df_clean = df_dated[df_dated['ticker'].isin(well_represented)].copy()
print(f"After coverage filter: {len(df_clean)} transcripts, {df_clean['ticker'].nunique()} tickers")

# Drop zero uncertainty rows
df_clean = df_clean[df_clean['uncertainty_score'] > 0].copy()
print(f"After dropping zero uncertainty: {len(df_clean)} transcripts")

# Save outputs
df_clean.to_csv("strux_scores_clean.csv", index=False)
manifest_clean = df_clean[["ticker", "date"]]
manifest_clean.to_csv("strux_manifest_clean.csv", index=False)
print("\nFiles saved: strux_scores_clean.csv, strux_manifest_clean.csv")

## 11. Clean Dataset — Sector Coverage

With 9 or more transcripts per ticker within 2018 to 2023, we retain approximately 50 tickers. Sector coverage is roughly 4 to 5 tickers per sector across 11 sectors. This is sufficient for the aggregate fixed-effects regression but insufficient for reliable sector-level subgroup regressions. Sector subgroup analysis has therefore been removed from the project scope and will be noted as a limitation in the paper.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean.groupby('sector')['ticker'].nunique().sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title("Unique Tickers per Sector (Clean Dataset)")

df_clean.boxplot(column='sentiment_score', by='sector', ax=axes[1], rot=45)
axes[1].set_title("Sentiment Score by Sector (Clean Dataset)")
axes[1].set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

print(f"\nTickers per sector:")
print(df_clean.groupby('sector')['ticker'].nunique().sort_values(ascending=False))

## 12. Spot Check — XOM Over Time

A visual check of sentiment and uncertainty scores over time for a single well-represented ticker. XOM has 28 transcripts spanning the full date range, making it a good candidate for visual inspection.

The notable dip in sentiment around early 2020 (COVID period) is consistent with the expectation that macro shocks affect executive tone. The general upward trend in sentiment post-2020 is also intuitive given the energy sector recovery.

In [ ]:
xom = df_clean[df_clean['ticker'] == 'XOM'].sort_values('date')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(xom['date'], xom['sentiment_score'], label='Sentiment', color='steelblue')
ax.plot(xom['date'], xom['uncertainty_score'] * 10, label='Uncertainty (x10)', color='darkorange')
ax.set_title("XOM — Sentiment and Uncertainty Over Time")
ax.set_xlabel("Date")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. EDA Summary

**Dataset scope after cleaning:**
- 632 tickers, 9,628 transcripts, 2018 to 2023
- 12 sectors with substantially more representation than the prior smaller dataset
- Top 5 sectors (Healthcare, Technology, Industrials, Financial Services, Consumer Cyclical) each have 73 or more tickers, making sector subgroup analysis viable

**Score quality:**
- Sentiment and uncertainty distributions look clean and sensible
- No null values in any score column
- No extreme sentiment scores (>0.9 or <-0.5) detected
- 8 zero uncertainty transcripts dropped before regression

**Key finding from EDA:**
- Q&A sentiment (mean 0.184) is meaningfully lower than prepared remarks sentiment (mean 0.346), suggesting executives are more guarded under analyst questioning. This will be tested as a supplementary regression.

**Implications for regression:**
- Sector subgroup analysis is now viable and should be added back to scope
- The 9-transcript coverage threshold will be tested as a robustness check at 6 and 12
- The clean dataset files are ready for Track B volatility computation